# 01 · text2ipc: from a patent text to IPC symbols

This notebook shows the public interface of `text2ipc` and evaluates it on 1,000
applications published by INPI Brazil (the *Revista da Propriedade Industrial*).

Prerequisites, run once from the repository root:

```bash
uv sync --all-extras
uv run t2ipc build --version 20260101 --lang PT   # Portuguese scheme, default model
uv run t2ipc build --version 20260101 --lang EN   # English scheme, for comparison
```

Everything below reads the Parquet tables under `data/` (or `$TEXT2IPC_HOME`).

In [1]:
from pathlib import Path

import pandas as pd

from text2ipc import IpcClassifier
from text2ipc.eval import evaluate, load_many

ROOT = Path.cwd() if (Path.cwd() / "evals").is_dir() else Path.cwd().parent
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)


def as_frame(matches):
    # One row per match; `path` is the full text the entry was embedded with
    # (section > class > subclass > group > subgroups), which is also the explanation.
    return pd.DataFrame(
        [
            {
                "symbol": m.pretty,
                "level": m.level,
                "score": round(m.score, 3),
                "sim": round(m.similarity, 3),
                "path": m.text,
            }
            for m in matches
        ]
    )

## 1. One classifier, one index

An `IpcClassifier` is bound to an IPC version, a scheme language and an embedding
model. Loading is lazy: the index (about 80k vectors) and the model load on the first
call. `lang="PT"` selects the scheme whose titles come from INPI's Portuguese
translation, the best choice for Portuguese text.

In [2]:
pt = IpcClassifier("20260101", lang="PT")
print(pt.version, pt.lang, pt.index.meta.model, len(pt.index), "entries")

20260101 PT st:intfloat/multilingual-e5-base 80145 entries


## 2. Classify a Portuguese abstract

`level` picks the hierarchy level of the answer: `section`, `class`, `subclass`,
`group` or `subgroup`. Results are ranked by a score that combines the entry's own
similarity with support from its ancestors and its subtree. The `path` column is the
whole chain of titles from the section down to the entry, exactly the text that was
embedded, so a result reads as its own explanation. After ranking, results that sit on
the same branch as a better-ranked one (an ancestor or a descendant of it) are
dropped, so each row is a distinct branch of the IPC tree.

In [3]:
# Modelled on an application from RPI 2100 (PI 0318341-6), which INPI filed under A62C 15/00.
abstract = "Aparelho para combate a incêndios com mangueira flexível reforçada e bico regulável."
as_frame(pt.classify(abstract, level="subclass", top_k=5))

/Users/jaimenms/code/personal/study-text-to-ipc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11885.43it/s]

,symbol,level,score,sim,path
0,A62C,subclass,0.854,0.854,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO
1,A62B,subclass,0.849,0.849,"NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > DISPOSITIVOS, APARELHOS OU MÉTODOS DE SALVAMENTO"
2,C06F,subclass,0.848,0.848,QUÍMICA; METALURGIA > EXPLOSIVOS; FÓSFOROS > FÓSFOROS; MANUFATURA DE FÓSFOROS
3,G04D,subclass,0.845,0.845,FÍSICA > HOROLOGIA > APARELHOS OU FERRAMENTAS ESPECIALMENTE ADAPTADOS PARA PRODUÇÃO OU MANUTENÇÃO DE RELÓGIOS DE PAREDE OU DE PULSO
4,F22G,subclass,0.843,0.843,ENGENHARIA MECÂNICA; ILUMINAÇÃO; AQUECIMENTO; ARMAS; EXPLOSÃO > GERAÇÃO DE VAPOR > SUPERAQUECIMENTO DE VAPOR


In [4]:
as_frame(pt.classify(abstract, level="group", top_k=5))

,symbol,level,score,sim,path
0,A62C 33/00,group,0.861,0.861,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Acessórios para mangueiras
1,A62C 35/00,group,0.857,0.857,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Equipamento instalado permanentemente
2,A62C 15/00,group,0.854,0.854,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Hand tools; Portable extinguishers; Hand fire-extinguishers > Extintores essencialmente do tipo mochila
3,A62C 29/00,group,0.852,0.852,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Barcos de combate ao fogo ou estruturas flutuantes similares
4,A62C 37/00,group,0.852,0.852,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Controle do equipamento de combate ao fogo


Compare with what INPI assigned, A62C 15/00 (fire-fighting apparatus with hoses).
The typical shape of a good answer today is a correct subclass near the top and the
assigned main group within the first few candidates; below group level the ranking
is noisy. Scores sit in a narrow band because e5 models compress cosine similarity,
so the order matters more than the absolute value.

## 3. As deep as the evidence supports

`level="auto"` walks down from the best subclasses along the best-scoring branch and
answers with the node on that walk whose own similarity is highest. Short or generic
texts stop high; specific ones reach subgroups.

In [5]:
for text in ["Aparelhos de combate a incêndios", abstract]:
    print(text[:60], "...")
    display(as_frame(pt.classify(text, level="auto", top_k=3)))

Aparelhos de combate a incêndios ...


,symbol,level,score,sim,path
0,A62D 1/06,subgroup,0.866,0.866,"NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > MEIOS QUÍMICOS PARA EXTINÇÃO DE INCÊNDIO; PROCESSOS PARA TORNAR INÓCUOS OU MENOS NOCIVOS OS AGENTES QUÍMICOS NOCIVOS EFETUANDO UMA TRANSFORMAÇÃO QUÍMICA; COMPOSIÇÃO DE MATERIAIS PARA REVESTIMENTOS OU ROUPAS PARA PROTEÇÃO CONTRA AGENTES QUÍMICOS NOCIVOS; COMPOSIÇÃO DE MATERIAIS PARA PARTES TRANSPARENTES DE MÁSCARAS CONTRA GÁS, RESPIRADORES, SACOS DE AR PARA RESPIRAÇÃO, OU CAPACETES; COMPOSIÇÃO DE MATERIAIS QUÍMICOS PARA USO EM APARELHOS RESPIRATÓRIOS > Composições para extinção de incêndios; Uso de substâncias químicas na extinção de incêndios > contendo componentes quimicamente reativos que produzem gás"
1,A62C 8/06,subgroup,0.865,0.865,"NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Hand tools; Portable extinguishers; Hand fire-extinguishers > Ferramentas manuais ou acessórios especialmente adaptados para combate ao fogo, p. ex. caixas de ferramentas > Cobertores de fogo"
2,G21C 9/04,subgroup,0.864,0.864,FÍSICA > FÍSICA NUCLEAR; ENGENHARIA NUCLEAR > REATORES NUCLEARES > Disposições de proteções de emergência associadas estruturalmente com o reator > Meios para combater incêndios


Aparelho para combate a incêndios com mangueira flexível ref ...


,symbol,level,score,sim,path
0,A62C 33/00,group,0.861,0.861,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Acessórios para mangueiras
1,A62B,subclass,0.849,0.849,"NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > DISPOSITIVOS, APARELHOS OU MÉTODOS DE SALVAMENTO"
2,C06F 5/00,group,0.848,0.848,QUÍMICA; METALURGIA > EXPLOSIVOS; FÓSFOROS > FÓSFOROS; MANUFATURA DE FÓSFOROS > Fósforos


## 4. Reading a result: the path an entry was embedded with

Every vector is the embedding of the entry's full path text, from the section down to
the entry itself. The scheme table keeps that path as a chain of symbols, so a result
can be explained.

In [6]:
from text2ipc.scheme import format_symbol

symbol = "A62C0015000000"  # A62C 15/00
print(" | ".join(format_symbol(s) for s in pt.index.scheme.path_of(symbol)))
print(pt.index.text_of(symbol))

A | A62 | A62C | A62C 15/00
NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO > Hand tools; Portable extinguishers; Hand fire-extinguishers > Extintores essencialmente do tipo mochila


## 5. Same question, English scheme

The embedder is multilingual, so a Portuguese query can be matched against the
English scheme too. It works, but the Portuguese scheme scores higher on the evals
below because nothing has to cross languages.

In [7]:
en = IpcClassifier("20260101", lang="EN")
as_frame(en.classify(abstract, level="group", top_k=5))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19473.34it/s]

,symbol,level,score,sim,path
0,H01C 11/00,group,0.843,0.843,ELECTRICITY > ELECTRIC ELEMENTS > RESISTORS > Non-adjustable liquid resistors
1,H01C 10/00,group,0.839,0.839,ELECTRICITY > ELECTRIC ELEMENTS > RESISTORS > Adjustable resistors
2,A62C 37/00,group,0.835,0.835,HUMAN NECESSITIES > LIFE-SAVING; FIRE-FIGHTING > FIRE-FIGHTING > Control of fire-fighting equipment
3,A62C 33/00,group,0.833,0.833,HUMAN NECESSITIES > LIFE-SAVING; FIRE-FIGHTING > FIRE-FIGHTING > Hose accessories
4,A62C 27/00,group,0.830,0.830,HUMAN NECESSITIES > LIFE-SAVING; FIRE-FIGHTING > FIRE-FIGHTING > Fire-fighting land vehicles


## 6. Multi-label answers with a score gap

`gap` keeps every result within that distance of the best score, so the number of
answers depends on how confident the ranking is. Cosine scores of e5 models live in a
narrow band, so gaps are small numbers.

In [8]:
as_frame(pt.classify(abstract, level="subclass", top_k=10, gap=0.005))

,symbol,level,score,sim,path
0,A62C,subclass,0.854,0.854,NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > COMBATE AO FOGO
1,A62B,subclass,0.849,0.849,"NECESSIDADES HUMANAS > SALVAMENTO; COMBATE AO FOGO > DISPOSITIVOS, APARELHOS OU MÉTODOS DE SALVAMENTO"


## 7. Evaluation on 1,000 INPI applications

`evals/` holds one JSONL per RPI issue (29 issues, about 12k applications). Each case
has the Portuguese title, the abstract when the issue printed one, and the IPC symbols
the office assigned. We draw 1,000 cases with a fixed seed, half with abstracts and
half title-only, and measure hit@k per hierarchy level: a hit when any gold symbol,
truncated to the level, appears among the top-k predictions.

In [9]:
cases = load_many(str(ROOT / "evals" / "rpi_*.jsonl"), sample=1000, seed=0)
with_abstract = [c for c in cases if c.abstract]
title_only = [c for c in cases if not c.abstract]
print(len(cases), "cases:", len(with_abstract), "with abstract,", len(title_only), "title-only")
print("issues:", sorted({c.id.split(":")[0] for c in cases})[:8], "...")

1000 cases: 500 with abstract, 500 title-only
issues: ['rpi2100', 'rpi2126', 'rpi2128', 'rpi2131', 'rpi2133', 'rpi2190', 'rpi2196', 'rpi2205'] ...


In [10]:
def run(clf, subset, name):
    result = evaluate(subset, lambda t: clf.classify(t, top_k=10), level="subgroup", top_k=10)
    frame = pd.DataFrame(result.rows())
    frame.insert(0, "run", name)
    return frame


frames = []
for label, subset in [("abstract", with_abstract), ("title-only", title_only)]:
    frames.append(run(pt, subset, f"PT scheme · {label}"))
    frames.append(run(en, subset, f"EN scheme · {label}"))
results = pd.concat(frames, ignore_index=True)
percent = [c for c in results.columns if c.startswith(("hit", "main"))]
shown = results.copy()
shown[percent] = shown[percent].apply(lambda col: col.map("{:.1%}".format))
shown["mrr"] = shown["mrr"].map("{:.3f}".format)
shown

,run,level,n,hit@1,hit@3,hit@5,hit@10,main@1,mrr
0,PT scheme · abstract,section,500,58.4%,76.2%,76.4%,76.4%,50.6%,0.668
1,PT scheme · abstract,class,500,34.2%,49.2%,53.0%,53.0%,29.6%,0.420
2,PT scheme · abstract,subclass,500,22.0%,34.0%,38.0%,38.2%,18.6%,0.283
3,PT scheme · abstract,group,500,10.0%,17.2%,19.8%,20.4%,6.8%,0.138
4,PT scheme · abstract,subgroup,500,1.2%,2.6%,3.4%,4.6%,0.6%,0.022
5,EN scheme · abstract,section,500,56.8%,74.8%,75.0%,75.0%,49.6%,0.655
6,EN scheme · abstract,class,500,34.2%,49.0%,51.0%,51.0%,28.6%,0.412
7,EN scheme · abstract,subclass,500,21.2%,31.2%,34.4%,34.4%,15.4%,0.263
8,EN scheme · abstract,group,500,11.4%,17.2%,19.6%,20.6%,6.2%,0.146
9,EN scheme · abstract,subgroup,500,2.4%,3.4%,4.6%,5.8%,0.4%,0.032


### Reading the table

- **Subclass** is the level a searcher usually needs first; **group** is what a
  classification officer assigns. Subgroups are noisy for a pure embedding approach.
- On titles the Portuguese scheme is clearly ahead (about 4 points at subclass@1 and
  at group@10). On abstracts the two schemes tie within the noise of 500 cases; the
  abstracts come from 2011 to 2015 issues, whose gold symbols predate several IPC
  revisions.
- Titles score higher than abstracts at rank 1: titles are keyword lists shaped like
  IPC entries, abstracts are prose. Concatenating title and abstract is an obvious
  next experiment.
- Gold symbols from 2011 to 2015 issues predate several IPC revisions; mapping them
  through WIPO's concordance list would make the abstract rows fairer.

In [11]:
pivot = results.pivot(index="level", columns="run", values="hit@10").loc[
    ["section", "class", "subclass", "group", "subgroup"]
]
pivot.apply(lambda col: col.map("{:.1%}".format))

run,EN scheme · abstract,EN scheme · title-only,PT scheme · abstract,PT scheme · title-only
level,,,,
section,75.0%,76.2%,76.4%,76.4%
class,51.0%,56.6%,53.0%,60.4%
subclass,34.4%,41.4%,38.2%,45.2%
group,20.6%,25.6%,20.4%,29.6%
subgroup,5.8%,7.2%,4.6%,8.8%


## 8. What to try next

1. A stronger multilingual embedder (`bge-m3`, `multilingual-e5-large`): the model,
   not the heuristics, is the bottleneck (`docs/evals.md`).
2. An LLM rerank over the 20 to 30 surviving candidates, with the path text as
   evidence.
3. `title + abstract` as the query text.
4. kNN over similar published applications (vote on their IPC symbols) as a second
   signal.